# nib — Colab evaluation

**This notebook contains no logic.** It clones, installs, mounts Drive, copies
one file to local disk, and calls scripts. Every decision lives in
`configs/base.yaml` and every line of code lives in the repository.

## What this run is for

On 2026-09-11 the project's style metric was found to be measuring the wrong
thing. A **real** line, by unquestionably the right writer, blurred by 0.8
pixels — damage that does not change whose handwriting it is — scored 12.2%
where the untouched line scored 96.8%. Every generative decoder produces
exactly that softness, so writer retrieval has been reporting sharpness as much
as style, and every style comparison this project has made is confounded by it.

`HWD` replaces it. Measured on the same damage it moves from 0.641 to 0.721, it
separates handwriting from a typeface by four and a half times, and it is what
Emuru's and Eruku's own papers report — so these figures will be the first this
project can set beside a published one.

**The scale, measured on our own data:**

```
0.641   two disjoint sets of real lines by the same writers
2.931   the same texts drawn in a typeface — no hand at all
```

## Run cells 1 to 7 in order

Cells 8 and 9 are optional. Anything below them is kept for reference and
should not be run without a reason.

## Before you start

Under `MyDrive/nib/` you need `cvl_lines_64.lmdb` (**127 MB, 9,142 records** —
the copy from `data/processed/upload/`, never the 8 GB one) and
`checkpoints/writer_embedder.pt`.

## 1. Clone and install

`torch` is deliberately absent — Colab's build is matched to its CUDA driver.

The `hwd` extra is new and it is a large install: the package imports every
score it owns at import time, so it pulls in gudhi, matplotlib, tiktoken and
scikit-learn. Two or three minutes.

**Expect a pip conflict warning about `gradio`.** Installing `transformers<5`
pulls `huggingface-hub` down to the range it needs and Colab's preinstalled
gradio wants something newer. Nothing here imports gradio. Cell 2 is the check
that decides.

In [ ]:
REPO_URL = "https://github.com/omritzabari/nib.git"

%cd /content
![ -d nib ] || git clone $REPO_URL nib
%cd /content/nib
!git pull --ff-only
!pip install -q -e ".[dev,track,models,hwd]"

## 2. What are we running on

**Stop here if either line is wrong.** A rebuilt Colab VM arrives without an
accelerator unless one is asked for, and generation on CPU is 220 seconds a
line — 18 hours for 300.

- `Tesla T4` must appear
- `torch` must say `+cu128`, not `+cpu`

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
import transformers

print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__, " <- must be 4.x")
assert transformers.__version__.startswith("4."), "5.x cannot load Emuru; see pyproject"

from nib.engine.metrics import hwd
print("hwd available:", hwd.available(), " <- must be True, or HWD is skipped")

## 3. Mount Drive and copy what the run needs

One sequential copy of one file. Reading the pack record-by-record over Drive
would leave the GPU waiting on network round-trips.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!mkdir -p /content/nib/data/processed /content/nib/checkpoints
!time cp /content/drive/MyDrive/nib/cvl_lines_64.lmdb /content/nib/data/processed/
!cp /content/drive/MyDrive/nib/checkpoints/writer_embedder.pt /content/nib/checkpoints/
!ls -lh /content/nib/data/processed/ /content/nib/checkpoints/

## 4. Is everything here

Two things will read as missing and both are correct: the **word pack**,
because one pack is required and this session needs lines; and the **raw CVL
images**, because 5 GB of sources are not copied to a VM that only reads a
127 MB pack. The only consequence is that CER cannot be re-measured here, and
it has already been measured where the sources are.

In [ ]:
%cd /content/nib
!python scripts/check_data.py

## 5. Check the harness — under a minute

The `fake` generator draws the target text in a typeface. Every number it
produces is meaningless and every shape is right, which is what a pipeline
check needs. It has caught an unexercised code path twice.

**What must appear**, or something below is not wired:

- square brackets after every figure — the 95% spreads
- an `HWD` block with a number, not `not measured`
- a line beginning `analysis`

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py --generator fake --samples 120 --device cuda

## 6. Emuru with a working style metric — the main run

About an hour. Emuru runs at 11 seconds a line.

This is the number the project has been missing. Everything before it measured
style with a ruler that responded to blur.

Watch `HWD` against the 0.641 floor. `truncated` should sit near 8%, which is
Emuru's known non-stopping rate and not a new problem.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py \
    --generator emuru \
    --samples 300 \
    --device cuda

## 7. Save, before anything else

An hour is too long to risk. Colab has reclaimed a VM mid-session on this
project already.

In [ ]:
!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_* /content/drive/MyDrive/nib/results/
!cp -r /content/nib/references /content/drive/MyDrive/nib/results/
!du -sh /content/drive/MyDrive/nib/results/*
!for f in /content/nib/outputs/eval_*/results.json; do echo "== $f"; cat "$f"; done

## 8. Eruku, for the comparison — optional, three hours

Eruku runs at 35 seconds a line: classifier-free guidance is two forward passes
per token rather than one, which is what buys its text fidelity and what costs
the time.

Worth doing because the Emuru-against-Eruku comparison made on 2026-09-10 was
decided by a metric that measures sharpness, and Eruku's output may simply be
softer. HWD will say. Run cell 7 again afterwards.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py \
    --generator eruku \
    --samples 300 \
    --device cuda

## 9. Compare what ran — no GPU, seconds

Reads what each run saved and reports whether the intervals separate at all.
Two figures whose intervals overlap are not a difference.

In [ ]:
%cd /content/nib
!python scripts/analyse_run.py outputs/eval_emuru_lines outputs/eval_eruku_lines

---

# Kept for reference — do not run without a reason

**Re-measuring the references** (`check_metrics.py`). They are measured and
committed in `references/`. CER cannot be computed here anyway, and the cell
downloads 1.4 GB of TrOCR in order to skip it.

**The guidance sweep.** Closed negative on 2026-09-10: cfg 1.0 / 1.25 / 2.0 gave
retrieval 10.0% / 5.3% / 6.7% with every interval overlapping, and cfg 2.0 was
clearly worse (15% truncated). Even cfg 1.0's upper bound sat below Emuru. It
would be worth revisiting only once HWD has replaced the metric that judged it.

**More than one style line.** `--style-refs 2` works (0.85x of real width);
`--style-refs 4` breaks Emuru (0.25x) because the prefix outgrows what its
stopping heuristic tolerates. Worth measuring at 2 with HWD, after cell 6
establishes the single-reference baseline.

In [ ]:
# %cd /content/nib
# !python scripts/check_metrics.py --pack data/processed/cvl_lines_64.lmdb --samples 300 --device cuda
# !python scripts/evaluate_generator.py --generator emuru --samples 300 --style-refs 2 --device cuda
# !python scripts/evaluate_generator.py --generator eruku-no-style-text --samples 300 --device cuda

## What to report back

- the `SUMMARY` block from each run, **with its intervals**
- the `HWD` figure against the 0.641 floor — this is the one that matters
- `truncated` and `empty outputs`
- anything that failed, with the full error text